Neighbourhood Verification

The Airbnb dataset contains listings from 40 neighbourhoods,
while the GeoJSON dataset contains 55 neighbourhood polygons.

All neighbourhoods present in the Airbnb dataset were found
within the GeoJSON dataset.

The remaining 15 GeoJSON neighbourhoods contain no Airbnb
listings and largely correspond to industrial, offshore,
airport, military, or sparsely populated areas.


In [2]:
import pandas as pd
import geopandas as gpd
from src.utils.paths import RAW_DATA_DIR, PROCESSED_DATA_DIR
from src.data.load import load_neighborhoods

In [3]:
df = pd.read_csv(PROCESSED_DATA_DIR / "listings_clean.csv")
gdf = gpd.read_file(RAW_DATA_DIR / "neighbourhoods.geojson")

In [4]:
df.head(5)

,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2011-01-29,1.0,1.0,1.0,False,5,10,True,True,Tampines,...,4.43,4.43,4.17,4.04,False,5,0,5,0,0.14
1,2011-01-29,1.0,1.0,1.0,False,5,10,True,True,Tampines,...,4.63,4.64,4.50,4.36,False,5,0,5,0,0.26
2,2011-12-20,1.0,1.0,1.0,False,7,8,True,True,Newton,...,4.50,4.66,4.52,4.39,False,7,1,6,0,0.79
3,2011-01-29,1.0,1.0,1.0,False,5,10,True,True,Tampines,...,4.76,4.58,4.66,4.44,True,5,0,5,0,0.36
4,2011-12-20,1.0,1.0,1.0,False,7,8,True,True,Newton,...,4.71,4.76,4.64,4.55,False,7,1,6,0,0.55


In [5]:
df["host_response_time"].value_counts(dropna=False)

host_response_time
0.0    1174
2.0     521
NaN     374
3.0     334
1.0     240
Name: count, dtype: int64

In [6]:
set([
    "host_location",
    "host_neighbourhood",
    "host_verifications",
    "license",
    "neighbourhood",
    "bathrooms_text",
]).intersection(df.columns)

set()

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2643 entries, 0 to 2642
Data columns (total 53 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   host_since                                    2643 non-null   str    
 1   host_response_time                            2269 non-null   float64
 2   host_response_rate                            2269 non-null   float64
 3   host_acceptance_rate                          2188 non-null   float64
 4   host_is_superhost                             2643 non-null   bool   
 5   host_listings_count                           2643 non-null   int64  
 6   host_total_listings_count                     2643 non-null   int64  
 7   host_has_profile_pic                          2643 non-null   bool   
 8   host_identity_verified                        2643 non-null   bool   
 9   neighbourhood_cleansed                        2643 non-null   str    
 10 

In [8]:
listings = df
neighborhoods = gdf

In [9]:
tabular_neighborhoods = set(
    listings["neighbourhood_cleansed"].unique()
)

spatial_neighborhoods = set(
    neighborhoods["neighbourhood"].unique()
)

print("Tabular:\t", len(tabular_neighborhoods))
print("Spatial:\t", len(spatial_neighborhoods))

Tabular:	 40
Spatial:	 55


In [10]:
tabular_neighborhoods - spatial_neighborhoods

set()

In [11]:
spatial_neighborhoods - tabular_neighborhoods

{'Boon Lay',
 'Changi Bay',
 'Lim Chu Kang',
 'Mandai',
 'Marina East',
 'North-Eastern Islands',
 'Paya Lebar',
 'Pioneer',
 'Seletar',
 'Simpang',
 'Straits View',
 'Sungei Kadut',
 'Tengah',
 'Tuas',
 'Western Islands'}

In [12]:
neighborhoods.columns

Index(['neighbourhood', 'neighbourhood_group', 'geometry'], dtype='str')

In [13]:
neighborhoods.head()

,neighbourhood,neighbourhood_group,geometry
0,Pasir Ris,East Region,"MULTIPOLYGON (((103.95322 1.38202, 103.9535 1...."
1,Seletar,North-East Region,"MULTIPOLYGON (((103.88691 1.42649, 103.88812 1..."
2,Sungei Kadut,North Region,"MULTIPOLYGON (((103.7644 1.44345, 103.76443 1...."
3,Orchard,Central Region,"MULTIPOLYGON (((103.84298 1.30001, 103.84294 1..."
4,Downtown Core,Central Region,"MULTIPOLYGON (((103.85942 1.29978, 103.85954 1..."


In [14]:
neighborhoods.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [15]:
neighborhoods_metric = neighborhoods.to_crs("EPSG:3414")